# AURAVOX — Neural GPU backend (free Colab T4)

Runs the real AI models (MusicGen for music + Bark for vocals) on a **free GPU** and exposes a public URL the AURAVOX web app can call.

**Steps:**
1. `Runtime → Change runtime type → Hardware accelerator: T4 GPU`, then Save.
2. (Optional but recommended) paste a free **ngrok token** in cell 3 for a **URL that never changes**. Get one at https://dashboard.ngrok.com/get-started/your-authtoken and claim your free static domain at https://dashboard.ngrok.com/cloud-edge/domains . Leave blank to use a random cloudflare URL instead.
3. `Runtime → Run all`.
4. Copy the printed **PUBLIC URL**, paste it into the web app's **Neural (GPU) mode** box, then Generate.

Keep this tab open — closing it (or the runtime timing out) stops the server. With a static ngrok domain the URL is the same next time; with cloudflare it changes each run.

> **Kaggle alternative (longer sessions):** Kaggle gives ~30 GPU-hours/week and longer runtimes than free Colab. Create a Kaggle Notebook, enable a GPU (Settings → Accelerator → GPU T4), turn **Internet** on, and paste the same 3 code cells below. ngrok works there too.

In [ ]:
# 1) Install deps (torch/torchaudio already present on Colab/Kaggle)
!pip -q install "transformers>=4.41" scipy fastapi "uvicorn[standard]" nest_asyncio pyngrok
import torch
print("CUDA available:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU — set the accelerator to T4 GPU")

In [ ]:
# 2) Download the AURAVOX backend server + the cloudflared tunnel binary (fallback)
BRANCH = "gh-pages"  # branch that holds ml/colab_server.py
REPO = "k58804494-pixel/ai-song-maker"
!wget -q -O colab_server.py https://raw.githubusercontent.com/{REPO}/{BRANCH}/ml/colab_server.py
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
print("downloaded server + cloudflared")

In [ ]:
# 3) Launch the server + a public tunnel, then print the URL to paste into the web app
#
#   ngrok (stable URL)  -> paste your token (and optional static domain) below.
#   leave NGROK_TOKEN blank -> a random https://*.trycloudflare.com URL is used.
NGROK_TOKEN  = ""   # e.g. "2abc...XYZ"  (https://dashboard.ngrok.com/get-started/your-authtoken)
NGROK_DOMAIN = ""   # e.g. "kamil-auravox.ngrok-free.app"  (your free reserved domain; optional)

import subprocess, sys, time, re
server = subprocess.Popen([sys.executable, "-m", "uvicorn", "colab_server:app", "--host", "0.0.0.0", "--port", "8000"])
time.sleep(4)

url = None
if NGROK_TOKEN.strip():
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_TOKEN.strip())
    opts = {"addr": 8000, "proto": "http"}
    if NGROK_DOMAIN.strip():
        opts["domain"] = NGROK_DOMAIN.strip()
    url = ngrok.connect(**opts).public_url
else:
    tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in tunnel.stdout:
        if "trycloudflare.com" in line:
            m = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
            if m:
                url = m.group(0)
                break

print("\n\n========================================")
print(" PUBLIC URL — paste this into the web app:")
print(" ", url)
print("========================================")
print("First Generate downloads the models (~2-4 GB) and can take a couple of minutes; later ones are fast.")
print("Leave this cell running. Stopping it shuts down the backend.")